# MinivLLM × Google Colab

这个 Notebook 会在 Colab 中运行 MinivLLM 的 Qwen3-0.6B 推理示例，适合没有本地 NVIDIA GPU 的 Mac 用户。

**运行前：** 在 Colab 顶部选择 `Runtime → Change runtime type → T4 GPU`，然后依次执行所有单元格。

Notebook 会完成以下工作：

1. 将 MinivLLM 克隆到 `/content/MinivLLM`
2. 安装推理所需的 Python 依赖
3. 检查 CUDA、PyTorch 和 Triton
4. 下载 Qwen3-0.6B 到 Colab 本地磁盘
5. 初始化自研推理引擎并生成回答

无需安装 `vllm`，也不会升级 Colab 预装的 CUDA 版 PyTorch。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# 如果要运行自己的修改版，请把这里替换为 fork 或私有镜像地址。
REPO_URL = "https://github.com/Wenyueh/MinivLLM.git"
REPO_DIR = Path("/content/MinivLLM")

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"{REPO_DIR} 已存在，但不是 Git 仓库。请先删除或改名该目录。")
else:
    print(f"Using existing repository: {REPO_DIR}")

SRC_DIR = REPO_DIR / "src"
if not SRC_DIR.exists():
    raise RuntimeError("未找到 src 目录，仓库可能没有克隆完整。")

os.chdir(REPO_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("Python module path:", SRC_DIR)

In [ ]:
# Colab 已提供匹配 CUDA 的 torch 和 triton，这里只安装本项目额外需要的包。
# 不使用 uv sync，因为它会安装本项目可选的 vllm 依赖。
%pip install -q transformers xxhash safetensors huggingface-hub

In [ ]:
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 CUDA GPU。请选择 Runtime → Change runtime type → T4 GPU，"
        "保存后重新连接运行时，再执行全部单元格。"
    )

try:
    import triton
except ImportError as exc:
    raise RuntimeError(
        "当前 Colab 环境缺少 Triton。请确认选择的是 GPU 运行时，并重启运行时后重试。"
    ) from exc

free_bytes, total_bytes = torch.cuda.mem_get_info()
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Triton:", triton.__version__)
print("GPU:", gpu_name)
print("Compute capability:", f"{capability[0]}.{capability[1]}")
print(f"Free VRAM: {free_bytes / 1024**3:.2f} GiB / {total_bytes / 1024**3:.2f} GiB")

if capability[0] < 7:
    raise RuntimeError("GPU 计算能力低于 7.0，当前 Triton kernel 可能无法运行。")
if free_bytes < 3 * 1024**3:
    raise RuntimeError("空闲显存不足 3 GiB，Qwen3-0.6B 推理可能会 OOM。")
if free_bytes < 5 * 1024**3:
    print("警告：显存较小，请把后面的 MAX_NEW_TOKENS 调低到 64。")

In [ ]:
from pathlib import Path
import os

# 如果 Hugging Face 下载失败，将 USE_HF_MIRROR 改为 True，重启运行时并重新运行此单元格。
USE_HF_MIRROR = False
if USE_HF_MIRROR:
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

os.environ.setdefault("HF_HOME", "/content/cache/huggingface")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-0.6B"
# 目录名必须保持 Qwen3-0.6B，ModelRunner 根据该名称选择模型实现。
MODEL_DIR = Path("/content/models/Qwen3-0.6B")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_ready = (MODEL_DIR / "config.json").exists() and any(MODEL_DIR.glob("*.safetensors"))
if not model_ready:
    print(f"Downloading {MODEL_ID} to {MODEL_DIR} ...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        ignore_patterns=["*.bin", "*.pt", "*.pth", "*.h5", "*.msgpack"],
    )
else:
    print("Model files already exist.")

print("Model directory:", MODEL_DIR)
print("Safetensors files:", [p.name for p in MODEL_DIR.glob("*.safetensors")])

In [ ]:
from transformers import AutoConfig, AutoTokenizer

from myvllm.engine.llm_engine import LLMEngine
from myvllm.sampling_parameters import SamplingParams

MAX_NEW_TOKENS = 96
MAX_MODEL_LENGTH = 384
PROMPT_TEXT = "用三句话介绍你自己，并说明什么是 PagedAttention。"

hf_config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

hidden_size = hf_config.hidden_size
num_heads = hf_config.num_attention_heads
num_kv_heads = getattr(hf_config, "num_key_value_heads", num_heads)
head_dim = getattr(hf_config, "head_dim", hidden_size // num_heads)

config = {
    "max_num_sequences": 4,
    "max_num_batched_tokens": 1024,
    "max_cached_blocks": 128,
    "block_size": 64,
    "world_size": 1,
    "model_name_or_path": str(MODEL_DIR),
    "enforce_eager": True,
    "vocab_size": hf_config.vocab_size,
    "hidden_size": hidden_size,
    "num_heads": num_heads,
    "head_dim": head_dim,
    "num_kv_heads": num_kv_heads,
    "intermediate_size": hf_config.intermediate_size,
    "num_layers": hf_config.num_hidden_layers,
    "tie_word_embeddings": getattr(hf_config, "tie_word_embeddings", True),
    "base": getattr(hf_config, "rope_theta", 1_000_000),
    "rms_norm_epsilon": getattr(hf_config, "rms_norm_eps", 1e-6),
    "qkv_bias": getattr(hf_config, "attention_bias", False),
    "scale": 1,
    "max_position": getattr(hf_config, "max_position_embeddings", 32768),
    "ffn_bias": getattr(hf_config, "mlp_bias", False),
    "max_num_batch_tokens": 1024,
    "max_model_length": MAX_MODEL_LENGTH,
    "gpu_memory_utilization": 0.75,
    "eos": tokenizer.eos_token_id,
}

sampling_params = SamplingParams(
    temperature=0.6,
    max_tokens=MAX_NEW_TOKENS,
    max_model_length=MAX_MODEL_LENGTH,
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": PROMPT_TEXT}],
    tokenize=False,
    add_generation_prompt=True,
)

print("Initializing MinivLLM...")
llm = LLMEngine(config=config)

print("Generating...")
outputs = llm.generate([prompt], sampling_params)

answer = tokenizer.decode(outputs["token_ids"][0], skip_special_tokens=True)
print("\nPrompt:", PROMPT_TEXT)
print("\nAnswer:", answer)
print("\nGenerated tokens:", len(outputs["token_ids"][0]))

## 常见问题

- `没有检测到 CUDA GPU`：在 Colab 菜单选择 `Runtime → Change runtime type → T4 GPU`，保存后点击 `Reconnect`，再从第一个单元格重新运行。
- Hugging Face 下载失败：把模型下载单元格中的 `USE_HF_MIRROR` 改为 `True`，重启运行时后重跑。
- `CUDA out of memory`：执行 `Runtime → Restart session`，把 `MAX_NEW_TOKENS` 改为 `64`，确保没有同时运行其他 GPU 程序。
- 长时间无输出：第一次加载权重和编译采样函数需要几分钟，以面板右上角的运行时状态为准。
- 需要运行自己的代码版本：先推送到 GitHub fork，再修改克隆单元格中的 `REPO_URL`；仅修改本地 Mac 文件不会自动出现在 Colab。
- Colab 的能力边界和后续 GPU 测试计划：见仓库中的 `docs/Colab_Learning_Scope_zh.md`。

Colab 每次断开后 `/content` 都可能被清空。如果需要反复使用，可以挂载 Google Drive 并修改 `MODEL_DIR`，但第一次运行暂时不需要。